# A/B Testing Analysis - E-commerce Dataset


## Data Loading

Customer behavior, events, and transaction data are combined to analyze an A/B experiment.


In [30]:
import pandas as pd

customers = pd.read_csv('data/customers.csv')
events = pd.read_csv('data/events.csv')
transactions = pd.read_csv('data/transactions.csv')
products = pd.read_csv('data/products.csv')
campaigns = pd.read_csv('data/campaigns.csv')

## Data Preparation

Conversion flag created (1 = purchase, 0 = no purchase)  
Events merged with transactions  
Missing values filled with 0  

This step prepares a unified dataset to identify which users converted.


In [31]:
customers.head()
events.head()
transactions.head()

,transaction_id,timestamp,customer_id,product_id,quantity,discount_applied,gross_revenue,campaign_id,refund_flag
0,1,2021-12-27 08:25:15,59540,1630.0,3,0.00,43.74,0,0
1,2,2023-06-06 21:14:26,54871,1901.0,3,0.00,174.78,21,0
2,3,2023-08-31 05:29:54,51818,1884.0,1,0.00,40.61,37,0
3,4,2022-06-26 20:33:46,18164,1114.0,2,0.15,68.76,13,0
4,5,2023-07-26 18:12:35,86915,408.0,1,0.00,14.64,4,0


In [32]:
# purchase yapanlar
transactions['converted'] = 1

In [33]:
df = events.merge(
    transactions[['customer_id', 'converted']],
    on='customer_id',
    how='left'
)

In [34]:
df['converted'] = df['converted'].fillna(0)

In [35]:
df[['customer_id','experiment_group','converted']].head()

,customer_id,experiment_group,converted
0,43812,Control,1.0
1,43812,Control,1.0
2,43812,Control,1.0
3,43812,Control,1.0
4,71340,Variant_A,1.0


## User-Level Dataset

Each user is assigned a single group and conversion flag.

Users may appear multiple times due to event-level data.  
To avoid duplication, we aggregate data at the user level.

In [36]:
df_user = df.groupby('customer_id').agg({
    'experiment_group': lambda x: x.value_counts().idxmax(),
    'converted': 'max'
}).reset_index()

## Validation

Check data consistency.

Ensure each user appears once and conversion is correctly assigned.

In [37]:
df_user.head()

,customer_id,experiment_group,converted
0,1,Control,0.0
1,2,Control,1.0
2,3,Control,0.0
3,4,Control,0.0
4,5,Control,0.0


## Conversion Rate

Compare groups.

This shows how each experiment group performs in terms of conversion.

In [38]:
df_user.groupby('experiment_group')['converted'].mean()

experiment_group
Control      0.641098
Variant_A    0.621058
Variant_B    0.619597
Name: converted, dtype: float64

In [39]:
df_user['experiment_group'].value_counts()

Control      96394
Variant_A     1871
Variant_B     1735
Name: experiment_group, dtype: int64

## Lift

Measure performance vs Control.

Lift shows how much a variant improves or worsens conversion compared to the baseline.

In [40]:
cr = df_user.groupby('experiment_group')['converted'].mean()

lift_A = (cr['Variant_A'] - cr['Control']) / cr['Control']
lift_B = (cr['Variant_B'] - cr['Control']) / cr['Control']

print("Lift Variant_A:", lift_A)
print("Lift Variant_B:", lift_B)

Lift Variant_A: -0.03125846006892442
Lift Variant_B: -0.03353847941694954


In [41]:
summary = pd.DataFrame({
    'Conversion Rate': cr,
})

summary.loc['Variant_A', 'Lift vs Control'] = lift_A
summary.loc['Variant_B', 'Lift vs Control'] = lift_B

summary

,Conversion Rate,Lift vs Control
experiment_group,,
Control,0.641098,NaN
Variant_A,0.621058,-0.031258
Variant_B,0.619597,-0.033538


## Key Findings

Control group has the highest conversion rate.

Both Variant_A and Variant_B perform worse than Control.

Differences are small but consistently negative for variants.

## Statistical Test

Check significance.

This test determines whether observed differences are real or due to randomness.

In [42]:
from statsmodels.stats.proportion import proportions_ztest

In [43]:
# başarı sayısı (converted = 1 olanlar)
success = [
    df_user[df_user['experiment_group'] == 'Control']['converted'].sum(),
    df_user[df_user['experiment_group'] == 'Variant_A']['converted'].sum()
]

# toplam user sayısı
nobs = [
    df_user[df_user['experiment_group'] == 'Control']['converted'].count(),
    df_user[df_user['experiment_group'] == 'Variant_A']['converted'].count()
]

stat, p_value = proportions_ztest(success, nobs)

print("Z-stat:", stat)
print("P-value:", p_value)

Z-stat: 1.7893811302193927
P-value: 0.07355345615852751


In [44]:
success = [
    df_user[df_user['experiment_group'] == 'Control']['converted'].sum(),
    df_user[df_user['experiment_group'] == 'Variant_B']['converted'].sum()
]

nobs = [
    df_user[df_user['experiment_group'] == 'Control']['converted'].count(),
    df_user[df_user['experiment_group'] == 'Variant_B']['converted'].count()
]

stat, p_value = proportions_ztest(success, nobs)

print("Z-stat:", stat)
print("P-value:", p_value)

Z-stat: 1.8500889236811235
P-value: 0.06430073409759263


## Conclusion

Control performs better, but differences are not statistically significant.

No rollout recommended.

Further testing with stronger variations is needed.

## Key Insight

Proper user-level aggregation is critical in A/B testing.

Incorrect grouping can lead to misleading results.